In [2]:
# Initialize environment and import dependencies

import os
import sys
import subprocess
import pandas as pd
import numpy as np
import polars as pl
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
import openpyxl

import elfen
from elfen.extractor import FUNCTION_MAP
os.environ["WN_DATA_DIR"] = "/mount/arbeitsdaten/e-delib/vecchiea/wn_data"
import wn

In [3]:
# Matplotlib & Seaborn Publication Configuration (for Book Chapter)
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "font.sans-serif": "DejaVu Sans",
    "font.family": "sans-serif",
    "figure.dpi": 300,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9
})

In [4]:
# 1. Load Excel Transcript & Annotations
# ---------------------------------------------------------
def load_excel_transcript_with_annotations(excel_path):
    df = pd.read_excel(excel_path)
    
    # Strip leading/trailing whitespace from column names
    df.columns = df.columns.str.strip()

    column_mapping = {}
    for col in df.columns:
        col_lower = col.lower()
        if col_lower in ['role', 'speaker']:
            column_mapping[col] = 'speaker'
        # Expanded checks for 'transcript', 'text', and common typos like 'transcipt'
        elif any(kw in col_lower for kw in ['transcript', 'transcipt', 'text']):
            column_mapping[col] = 'text'
        elif 'affiliation' in col_lower:
            column_mapping[col] = 'affiliation'
        elif 'dominance' in col_lower:
            column_mapping[col] = 'dominance'
        elif 'rupture typ' in col_lower:
            column_mapping[col] = 'rupture_type'
        elif 'rupture strength' in col_lower:
            column_mapping[col] = 'rupture_strength'
        elif 'uptake' in col_lower:
            column_mapping[col] = 'disc_uptake'
        elif 'elaboration' in col_lower:
            column_mapping[col] = 'disc_elaboration'
        elif 'grounding moves' in col_lower:
            column_mapping[col] = 'disc_grounding'
        elif 'metacommunication' in col_lower:
            column_mapping[col] = 'disc_metacommunication'
        elif 'relevance' in col_lower:
            column_mapping[col] = 'disc_relevance'
            
    df = df.rename(columns=column_mapping)
    df['speaker'] = df['speaker'].astype(str).str.strip().replace({'Therapaist': 'Therapist'})
    df['turn_id'] = range(1, len(df) + 1)
    
    # Standardize missing numeric annotation values
    for col in ['affiliation', 'dominance', 'rupture_strength', 'disc_relevance', 'disc_metacommunication']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    label_map = {
    '': 'Baseline',
    'none': 'Baseline',
    'baseline': 'Baseline',
    'withdraw': 'Withdrawal',
    'withdrawal': 'Withdrawal',
    'withdrawl': 'Withdrawal',
    'withdrraw': 'Withdrawal',
    'widrraw': 'Withdrawal',
    'confront': 'Confrontation',
    'confrontation': 'Confrontation',
    'repair': 'Repair',
    'resolution': 'Repair',
    'metacommunication': 'Repair',
    }

    if 'rupture_type' not in df.columns:
        raise ValueError("Missing 'rupture_type' column.")

    labels = (
        df['rupture_type']
        .fillna('')
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Flag unexpected labels rather than silently treating them as baseline.
    unknown = sorted(set(labels) - set(label_map))
    if unknown:
        raise ValueError(f"Unrecognized rupture labels: {unknown}")

    df['interaction_state'] = labels.map(label_map)

    # Preserve pooled rupture comparisons for Figure 1.
    df['phase'] = df['interaction_state'].replace({
        'Withdrawal': 'Rupture',
        'Confrontation': 'Rupture',
    })

    return df

# Set data paths
excel_file_path = "/mount/arbeitsdaten/e-delib/vecchiea/data/answering_machine/transcript_alcohol_mistral_humanrating_discourse.xlsx"
excel_path = Path(excel_file_path)
output_dir = excel_path.parent / excel_path.stem
output_dir.mkdir(parents=True, exist_ok=True)

df = load_excel_transcript_with_annotations(excel_file_path)

# Safe dynamic detection of the text column
if 'text' in df.columns:
    text_col = 'text'
else:
    # Fallback: select the first non-annotation string column
    non_text_cols = ['speaker', 'affiliation', 'dominance', 'rupture_type', 'rupture_strength', 'turn_id', 'phase', 'disc_relevance', 'disc_metacommunication']
    candidates = [c for c in df.columns if c not in non_text_cols]
    text_col = candidates[0] if candidates else df.columns[1]

# Clean empty rows safely
df[text_col] = df[text_col].fillna("").astype(str)
df = df[df[text_col].str.strip() != ""].reset_index(drop=True)

In [15]:
# 2. Discourse Annotation Preparation
# ---------------------------------------------------------

GROUNDING_MOVES = ['Clarification request', 'Reformulation', 'Confirmation',
                   'Confirmation request', 'Correction', 'Self-correction']
DISCOURSE_CATEGORIES = {
    'disc_relevance': [-1, 0.5, 1],
    'disc_uptake': ['Acknowledges', 'Accepts', 'Qualifies', 'Rejects'],
    'disc_elaboration': ['Minimal', 'Adds information', 'Explanation'],
    'disc_metacommunication': [0, 1],
}

def prepare_discourse(data):
    # Add derived columns only. Existing names and values remain unchanged.
    data = data.copy()
    required = ['disc_relevance', 'disc_uptake', 'disc_elaboration',
                'disc_grounding', 'disc_metacommunication']
    missing = [c for c in required if c not in data]
    if missing:
        raise ValueError(f'Missing discourse columns: {missing}')
    for field, allowed in [('disc_relevance', [-1, 0, 0.5, 1]),
                           ('disc_metacommunication', [0, 1])]:
        if not data[field].dropna().isin(allowed).all():
            raise ValueError(f'Unexpected values in {field}; expected {allowed}.')

    # pd.read_excel() can turn literal None/N/A into NaN. Read only the source
    # annotation fields again to recover those distinctions for derived measures.
    # Your loading cell and its original columns are not altered.
    source = pd.read_excel(excel_file_path, keep_default_na=False)
    source.columns = source.columns.str.strip()
    if data['turn_id'].duplicated().any() or not data['turn_id'].isin(range(1, len(source)+1)).all():
        raise ValueError('Turn IDs must match the row numbering from your loading cell.')
    def source_annotation(fragment):
        columns = [c for c in source if fragment in c.lower()]
        if len(columns) != 1:
            raise ValueError(f'Expected exactly one {fragment} column in Excel.')
        values = source[columns[0]].copy()
        values.index = np.arange(1, len(source)+1)
        return data['turn_id'].map(values).fillna('').astype(str).str.strip()

    for field, fragment in [('disc_uptake','uptake'), ('disc_elaboration','elaboration')]:
        values = source_annotation(fragment)
        unexpected = set(values) - set(DISCOURSE_CATEGORIES[field]) - {'', 'N/A', 'None'}
        if unexpected:
            raise ValueError(f'Unrecognized {field} values: {sorted(unexpected)}')
        # None is not a documented uptake category; retain it as missing.
        data[field + '_not_applicable'] = values.eq('N/A')

    parsed = []
    for value in source_annotation('grounding moves'):
        if not value:
            parsed.append(None)
        elif value.lower() == 'none':
            parsed.append(set())
        else:
            lookup = {m.lower(): m for m in GROUNDING_MOVES}
            tokens = {x.strip().lower() for x in value.split(';')}
            if tokens - set(lookup):
                raise ValueError(f'Unrecognized grounding moves: {tokens - set(lookup)}')
            parsed.append({lookup[x] for x in tokens})
    data['disc_grounding_status'] = ['Missing' if x is None else 'Present' if x else 'None' for x in parsed]
    for move in GROUNDING_MOVES:
        column = 'disc_grounding_' + move.lower().replace('-', '_').replace(' ', '_')
        data[column] = [np.nan if x is None else float(move in x) for x in parsed]
    data['disc_grounding_any'] = [np.nan if x is None else float(bool(x)) for x in parsed]
    data['disc_meta_present'] = data['disc_metacommunication']
    return data

def discourse_not_applicable(data, field):
    if field == 'disc_relevance':
        return data[field].eq(0)
    if field + '_not_applicable' in data:
        return data[field + '_not_applicable']
    return data[field].eq('N/A')

df = prepare_discourse(df)
# If extraction has already run, append only new discourse-derived columns by ID.
if 'df_results' in globals():
    if df_results['turn_id'].duplicated().any() or set(df_results['turn_id']) != set(df['turn_id']):
        raise ValueError('df_results contains different turns. Rerun loading and extraction for this workbook.')
    discourse_columns_to_add = [
        column for column in df.columns
        if column.startswith('disc_') and column not in df_results.columns
    ]
    if discourse_columns_to_add:
        discourse_additions = (
            df.set_index('turn_id')[discourse_columns_to_add]
            .reindex(df_results['turn_id'])
        )
        discourse_additions.index = df_results.index
        df_results = pd.concat(
            [df_results, discourse_additions], axis=1
        ).copy()

def discourse_distributions(data, groups):
    rows = []
    variables = dict(DISCOURSE_CATEGORIES)
    variables['disc_grounding_any'] = [0., 1.]
    for move in GROUNDING_MOVES:
        variables['disc_grounding_' + move.lower().replace('-', '_').replace(' ', '_')] = [0., 1.]
    for keys, group in data.groupby(groups, dropna=False, sort=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        for field, categories in variables.items():
            valid = group[field].isin(categories)
            n_valid = int(valid.sum())
            for category in categories:
                count = int(group[field].eq(category).sum())
                rows.append(dict(zip(groups, keys), annotation=field, category=category,
                                 n_category=count, n_valid=n_valid, n_total=len(group),
                                 n_missing=int((~valid & ~discourse_not_applicable(group, field)).sum()),
                                 n_not_applicable=int(discourse_not_applicable(group, field).sum()),
                                 percent=100 * count / n_valid if n_valid else np.nan))
    return pd.DataFrame(rows)

def adjacent_responses(data):
    # Retain every adjacent T→P pair. Never search ahead past intervening turns.
    rows = []
    fields = ['text', 'interaction_state', 'disc_relevance', 'disc_uptake',
              'disc_elaboration', 'disc_metacommunication', 'disc_grounding_status',
              'disc_grounding_any'] + ['disc_grounding_' + m.lower().replace('-', '_').replace(' ', '_') for m in GROUNDING_MOVES]
    for i in range(1, len(data)):
        therapist, patient = data.iloc[i-1], data.iloc[i]
        if therapist['speaker'] != 'Therapist' or patient['speaker'] != 'Patient' or patient['turn_id'] != therapist['turn_id'] + 1:
            continue
        row = {'therapist_turn_id': therapist['turn_id'], 'patient_turn_id': patient['turn_id']}
        for field in fields:
            row['therapist_' + field] = therapist[field]
            row['patient_' + field] = patient[field]
        rows.append(row)
    columns = ['therapist_turn_id', 'patient_turn_id'] + [prefix + f for f in fields for prefix in ['therapist_', 'patient_']]
    return pd.DataFrame(rows, columns=columns)

def sequential_summary(pairs):
    rows = []
    predictors = ['therapist_disc_metacommunication', 'therapist_disc_grounding_any']
    predictors += ['therapist_disc_grounding_' + m.lower().replace('-', '_').replace(' ', '_') for m in GROUNDING_MOVES]
    for predictor in predictors:
        levels = [0, 1]
        for level in levels:
            subset = pairs.loc[pairs[predictor].eq(level)]
            for outcome, categories in [('patient_disc_uptake', DISCOURSE_CATEGORIES['disc_uptake']),
                                        ('patient_interaction_state', ['Baseline', 'Withdrawal', 'Confrontation', 'Repair'])]:
                n_valid = int(subset[outcome].isin(categories).sum())
                for category in categories:
                    n = int(subset[outcome].eq(category).sum())
                    rows.append({'therapist_move': predictor, 'move_value': level,
                                 'patient_outcome': outcome, 'category': category,
                                 'n_pairs': len(subset), 'n_valid_outcomes': n_valid, 'n_category': n,
                                 'n_missing_predictor_all_pairs': int(pairs[predictor].isna().sum()),
                                 'percent': 100*n/n_valid if n_valid else np.nan})
    return pd.DataFrame(rows)

def rupture_windows(data):
    # P(rupture)→T(response)→P(next turn), with text for qualitative checking.
    rows = []
    for i, first in data.iterrows():
        if first['speaker'] != 'Patient' or first['phase'] != 'Rupture':
            continue
        row = {'rupture_turn_id': first['turn_id'], 'rupture_state': first['interaction_state'],
               'rupture_text': first['text'], 'complete_window': False}
        if i+2 < len(data):
            middle, last = data.iloc[i+1], data.iloc[i+2]
            if middle['speaker']=='Therapist' and last['speaker']=='Patient' and middle['turn_id']==first['turn_id']+1 and last['turn_id']==first['turn_id']+2:
                row.update(complete_window=True, therapist_turn_id=middle['turn_id'],
                           therapist_text=middle['text'], therapist_metacommunication=middle['disc_metacommunication'],
                           therapist_grounding=middle['disc_grounding'], next_patient_turn_id=last['turn_id'],
                           next_patient_text=last['text'], next_patient_uptake=last['disc_uptake'],
                           next_patient_state=last['interaction_state'])
        rows.append(row)
    return pd.DataFrame(rows, columns=['rupture_turn_id','rupture_state','rupture_text','complete_window',
                                     'therapist_turn_id','therapist_text','therapist_metacommunication',
                                     'therapist_grounding','next_patient_turn_id','next_patient_text',
                                     'next_patient_uptake','next_patient_state'])

discourse_by_speaker = discourse_distributions(df, ['speaker'])
discourse_by_state = discourse_distributions(df, ['speaker', 'interaction_state'])
pairs = adjacent_responses(df)
sequence_summary = sequential_summary(pairs)
windows = rupture_windows(df)

for name, table in {'discourse_turn_data': df, 'discourse_by_speaker': discourse_by_speaker,
                    'discourse_by_clinical_state': discourse_by_state,
                    'therapist_patient_pairs': pairs, 'sequential_response_summary': sequence_summary,
                    'rupture_response_windows': windows}.items():
    table.to_csv(output_dir / f'{name}.csv', index=False)

# Audit includes constant categories: absence of variation is a result, not an error.
audit = []

for speaker, group in df.groupby('speaker'):
    for field in list(DISCOURSE_CATEGORIES) + ['disc_grounding_any']:
        allowed = DISCOURSE_CATEGORIES.get(field, [0, 1])
        valid_mask = group[field].isin(allowed)
        valid = group.loc[valid_mask, field]
        audit.append({'speaker': speaker, 'annotation': field, 'n_total': len(group),
                      'n_valid': len(valid), 'n_missing': int((~valid_mask & ~discourse_not_applicable(group, field)).sum()),
                      'n_not_applicable': int(discourse_not_applicable(group, field).sum()),
                      'n_distinct_valid': valid.nunique(), 'constant': valid.nunique() == 1})
audit = pd.DataFrame(audit)
audit.to_csv(output_dir / 'discourse_annotation_audit.csv', index=False)
print(audit.to_string(index=False))
print(f'{len(pairs)} adjacent therapist–patient pairs; {len(windows)} rupture response windows.')
print('Blank discourse cells remain missing. Use literal None for observed absence of grounding; N/A for inapplicable uptake.')

# Figure 4: a fixed selection of discourse categories, never ranked by effect size.
profile = discourse_by_state.loc[discourse_by_state['speaker'].eq('Patient')].copy()
labels = {('disc_uptake','Accepts'):'Accepts', ('disc_uptake','Qualifies'):'Qualifies',
          ('disc_uptake','Rejects'):'Rejects', ('disc_uptake','Acknowledges'):'Acknowledges',
          ('disc_relevance',1):'Direct relevance', ('disc_elaboration','Explanation'):'Explanation',
          ('disc_metacommunication',1):'Metacommunication', ('disc_grounding_any',1.):'Any grounding move'}
profile['measure'] = [labels.get((a,c)) for a,c in zip(profile.annotation, profile.category)]
profile = profile.dropna(subset=['measure'])
profile.to_csv(output_dir / 'figure4_plot_data.csv', index=False)
fig, ax = plt.subplots(figsize=(10, 6))
states = [s for s in ['Baseline','Withdrawal','Confrontation','Repair'] if s in set(profile.interaction_state)]
height = 0.8 / max(1, len(states))
for j, state in enumerate(states):
    part = profile.loc[profile.interaction_state.eq(state)].set_index('measure').reindex(labels.values())
    y = np.arange(len(labels)) + (j-(len(states)-1)/2)*height
    ax.barh(y, part['percent'], height=height, label=state)
    for position, (_, row) in zip(y, part.iterrows()):
        if pd.notna(row.n_valid):
            label = f'{int(row.n_category)}/{int(row.n_valid)}' if row.n_valid else ''
            ax.text((row.percent if pd.notna(row.percent) else 0)+1, position, label, va='center', fontsize=5)
ax.set_yticks(range(len(labels)), labels.values()); ax.invert_yaxis()
ax.set_xlim(0,125); ax.set_xticks([0,25,50,75,100]); ax.set_xlabel('Patient turns with discourse feature (%)')
ax.set_title('Figure 4: Patient Discourse Features by Clinical State', fontweight='bold')
ax.legend(loc='upper left', bbox_to_anchor=(1.01,1), fontsize=9)
fig.text(.01,.01,'Labels show count / valid denominator. Missing and N/A excluded. Categories can overlap across dimensions.',fontsize=8)
fig.tight_layout(rect=(0,.045,1,1)); fig.savefig(output_dir/'figure4_discourse_profile.png',dpi=200); plt.close(fig)

# Figure 5: temporal association, not evidence that a move caused repair.
seqplot = sequence_summary.loc[sequence_summary.therapist_move.eq('therapist_disc_metacommunication') & sequence_summary.patient_outcome.eq('patient_disc_uptake')].copy()
seqplot.to_csv(output_dir/'figure5_plot_data.csv', index=False)
fig, ax = plt.subplots(figsize=(8,4.5))
cats = DISCOURSE_CATEGORIES['disc_uptake']
for j, level in enumerate([0, 1]):
    part = seqplot.loc[seqplot.move_value.eq(level)].set_index('category').reindex(cats)
    ax.bar(np.arange(len(cats))+(j-.5)*.36, part.percent, width=.36,label="Therapist metacommunication: " + ["absent", "present"][level])
    for k, row in enumerate(part.itertuples()):
        ax.text(k+(j-.5)*.36,(row.percent if pd.notna(row.percent) else 0)+1,
                f'{row.n_category}/{row.n_valid_outcomes}',ha='center',fontsize=8)
ax.set_xticks(range(len(cats)),cats); ax.set_ylim(0,115)
ax.set_ylabel('Percent of observed, applicable next-patient uptake'); ax.legend(fontsize=8)
ax.set_title("Figure 5: Patient Uptake Following Therapist Metacommunication", fontweight='bold')
fig.text(.01,.01,'Adjacent T→P pairs only. Counts / valid outcomes shown. Temporal association does not establish repair or causation.',fontsize=8)
fig.tight_layout(rect=(0,.05,1,1)); fig.savefig(output_dir/'figure5_sequential_uptake.png',dpi=200); plt.close(fig)



  speaker             annotation  n_total  n_valid  n_missing  n_not_applicable  n_distinct_valid  constant
  Patient         disc_relevance       16       15          0                 1                 1      True
  Patient            disc_uptake       16       14          2                 0                 3     False
  Patient       disc_elaboration       16       16          0                 0                 1      True
  Patient disc_metacommunication       16       16          0                 0                 2     False
  Patient     disc_grounding_any       16       13          3                 0                 2     False
Therapist         disc_relevance       15       15          0                 0                 2     False
Therapist            disc_uptake       15       14          1                 0                 3     False
Therapist       disc_elaboration       15       15          0                 0                 2     False
Therapist disc_metacommunica

In [16]:
# 3. Extract Feature Sets with elfen & Compute Sequential Cues
# ---------------------------------------------------------
print("Extracting linguistic features with elfen...")
df_polars = pl.from_pandas(df)

extractor = elfen.Extractor(
    data=df_polars,
    text_column=text_col,
    language="en",
    model="en_core_web_sm"
)
extractor.extract_features()
df_results = extractor.data.to_pandas()

# Calculate patient turn length relative to the preceding therapist turn.
df_results = df_results.sort_values('turn_id').reset_index(drop=True)

df_results['word_count'] = (
    df_results[text_col].fillna('').astype(str).str.split().str.len()
)

previous_count = df_results['word_count'].shift(1)
previous_speaker = df_results['speaker'].shift(1)

valid_pair = (
    df_results['speaker'].eq('Patient')
    & previous_speaker.eq('Therapist')
    & previous_count.gt(0)
)

# Leave missing when no valid preceding therapist turn exists.
df_results['prior_word_count'] = previous_count.where(valid_pair)
df_results['turn_length_ratio'] = (
    df_results['word_count'] / previous_count.where(valid_pair)
)

annotation_cols = [
    'turn_id', 'speaker', text_col, 'phase',
    'affiliation', 'dominance', 'rupture_type', 'rupture_strength'
]

patient_df = df_results[df_results['speaker'] == 'Patient'].copy()

# Fixed feature selection, used consistently across transcripts.
SELECTED_FEATURES = [
    # Surface and turn structure
    'n_tokens',
    'n_sentences',
    'tokens_per_sentence',
    'n_characters',
    'avg_word_length',
    'n_types',
    'n_lemmas',

    # Grammatical composition
    'n_noun',
    'n_verb',
    'n_adj',
    'n_adv',
    'n_aux',
    'n_PRON_Person_1',
    'n_PRON_Person_2',
    'n_PRON_Person_3',
    'n_PRON_PronType_Ind',
    'n_PRON_PronType_Prs',
    'n_PRON_Gender_Neut',
    'n_VERB_Tense_Pres',
    'n_VERB_Tense_Past',
    'n_VERB_Aspect_Prog',
    'n_VERB_VerbForm_Part',
    'n_dependency_nsubj',
    'n_dependency_dobj',
    'n_dependency_dative',
    'n_dependency_advcl',
    'n_dependency_ccomp',
    'n_dependency_appos',
    'n_dependency_conj',
    'n_dependency_amod',
    'n_dependency_advmod',
    'tree_depth',
    'tree_width',
    'tree_branching',
    'pos_variability',
    'lexical_density',

    # Lexical diversity
    'ttr',
    'rttr',
    'cttr',
    'herdan_c',
    'giroud_index',
    'mtld',
    'mattr',
    'msttr',
    'hdd',
    'yule_k',
    'simpsons_d',
    'sichel_s',
#    'global_sichel_s',
#    'n_hapax_legomena',
#    'n_hapax_dislegomena',
#    'n_global_token_hapax_legomena',
#    'n_global_lemma_hapax_legomena',
    'n_global_token_hapax_dislegomena',
    'n_global_lemma_hapax_dislegomena',

    # Semantic and lexical accessibility
    'n_hedges',
    'avg_n_synsets',
    'avg_n_synsets_noun',
    'avg_n_synsets_verb',
    'avg_n_synsets_adj',
    'avg_n_synsets_adv',
    'n_low_synsets',
    'n_high_synsets',
    'n_low_synsets_noun',
    'n_high_synsets_noun',
    'n_low_synsets_verb',
    'n_high_synsets_verb',

    # Affective vocabulary
    'n_positive_sentiment',
    'n_negative_sentiment',
    'sentiment_score',
    'avg_valence',
    'avg_arousal',
    'avg_dominance',
    'n_low_valence',
    'n_high_valence',
    'n_low_arousal',
    'n_high_arousal',
    'n_low_dominance',
    'n_high_dominance',
    'avg_intensity_anger',
    'n_low_intensity_anger',
    'n_high_intensity_anger',

    # Psycholinguistic associations
    'avg_concreteness',
    'avg_sd_concreteness',
    'n_low_concreteness',
    'n_high_concreteness',
    'avg_socialness',
    'avg_sd_socialness',
    'n_low_socialness',
    'n_high_socialness',
    'avg_Auditory_sensorimotor',
    'avg_Interoceptive_sensorimotor',
    'avg_Hand_arm_sensorimotor',
    'avg_Mouth_sensorimotor',
    'avg_Torso_sensorimotor',
    'avg_sd_Auditory_sensorimotor',
    'avg_sd_Interoceptive_sensorimotor',
    'avg_sd_Hand_arm_sensorimotor',
    'avg_sd_Mouth_sensorimotor',
    'avg_sd_Torso_sensorimotor',
    'avg_iconicity',
    'avg_sd_iconicity',

    # Readability and information structure
    'flesch_reading_ease',
    'flesch_kincaid_grade',
    'gunning_fog',
    'ari',
    'lix',
    'compressibility',
    'entropy',
]

missing = [
    feature for feature in SELECTED_FEATURES
    if feature not in df_results.columns
]
if missing:
    raise ValueError(f"Selected features missing from ELFEN output: {missing}")

# Convert count features to frequencies per 100 tokens.
# Keep turn size and sentence count as raw counts.
COUNT_FEATURES = [
    'n_PRON_Person_1',
    'n_PRON_Person_2',
    'n_PRON_Person_3',
    'n_PRON_PronType_Ind',
    'n_PRON_PronType_Prs',
    'n_PRON_Gender_Neut',
    'n_VERB_Tense_Pres',
    'n_VERB_Tense_Past',
    'n_VERB_Aspect_Prog',
    'n_VERB_VerbForm_Part',
    'n_dependency_nsubj',
    'n_dependency_dobj',
    'n_dependency_dative',
    'n_dependency_advcl',
    'n_dependency_ccomp',
    'n_dependency_appos',
    'n_dependency_conj',
    'n_dependency_amod',
    'n_dependency_advmod',
    'n_noun',
    'n_adj',
    'n_positive_sentiment',
    'n_negative_sentiment',
    'n_low_synsets',
    'n_high_synsets',
    'n_low_synsets_noun',
    'n_high_synsets_noun',
    'n_low_synsets_verb',
    'n_high_synsets_verb',
]
    
token_count = pd.to_numeric(
    df_results['n_tokens'], errors='coerce'
)
denominator = token_count.where(token_count > 0)

elfen_feature_cols = []

for feature in SELECTED_FEATURES:
    values = pd.to_numeric(
        df_results[feature], errors='coerce'
    ).replace([np.inf, -np.inf], np.nan)

    if feature in COUNT_FEATURES:
        analysis_column = f'{feature}_per_100_tokens'
        df_results[analysis_column] = 100 * values / denominator
    else:
        analysis_column = feature
        df_results[analysis_column] = values

    elfen_feature_cols.append(analysis_column)

# Rebuild after adding normalized columns.
patient_df = df_results.loc[
    df_results['speaker'].eq('Patient')
].copy()

print(f"Selected {len(elfen_feature_cols)} features for analysis.")


Extracting linguistic features with elfen...
Extracting raw_sequence_length...
Extracting n_tokens...
Extracting n_sentences...
Extracting n_tokens_per_sentence...
Extracting n_characters...
Extracting avg_word_length...
Extracting n_types...
Extracting n_long_words...
Extracting n_lemmas...
Extracting n_per_morph_feature...
Extracting tree_width...
Extracting tree_depth...
Extracting tree_branching...
Extracting n_noun_chunks...
Extracting n_per_dependency_type...
Extracting sentiment_score...
Extracting n_negative_sentiment...
Extracting n_positive_sentiment...
Extracting avg_valence...
Extracting n_low_valence...
Extracting n_high_valence...
Extracting avg_arousal...
Extracting n_low_arousal...
Extracting n_high_arousal...
Extracting avg_dominance...
Extracting n_low_dominance...
Extracting n_high_dominance...
Extracting avg_emotion_intensity...
Extracting n_low_intensity...
Extracting n_high_intensity...
Extracting compressibility...
Extracting entropy...
Extracting lemma_token_rat

In [ ]:
# 4. FIGURE 1: Linguistic Feature Shifts Across Interaction Phases
# ---------------------------------------------------------

# Replace the entire existing Figure 1 cell with this cell.

FEATURE_DISPLAY_NAMES = {
    'n_tokens': 'Turn length',
    'n_sentences': 'Sentence count',
    'tokens_per_sentence': 'Tokens per sentence',
    'n_characters': 'Character count',
    'avg_word_length': 'Average word length',
    'n_types': 'Unique word types',
    'n_lemmas': 'Lemma count',
    'n_noun': 'Noun count',
    'n_verb': 'Verb count',
    'n_adj': 'Adjective count',
    'n_adv': 'Adverb count',
    'n_aux': 'Auxiliary-verb count',
    'n_PRON_Person_1': '1-person pronouns',
    'n_PRON_Person_2': '2-person pronouns',
    'n_PRON_Person_3': '3-person pronouns',
    'n_PRON_PronType_Ind': 'Indefinite pronouns',
    'n_PRON_PronType_Prs': 'Personal pronouns',
    'n_PRON_Gender_Neut': 'Neuter pronouns',
    'n_VERB_Tense_Pres': 'Present-tense verbs',
    'n_VERB_Tense_Past': 'Past-tense verbs',
    'n_VERB_Aspect_Prog': 'Progressive verbs',
    'n_VERB_VerbForm_Part': 'Participles',
    'n_dependency_nsubj': 'Nominal subjects',
    'n_dependency_dobj': 'Direct objects',
    'n_dependency_dative': 'Dative/indirect objects',
    'n_dependency_advcl': 'Adverbial clauses',
    'n_dependency_ccomp': 'Clausal complements',
    'n_dependency_appos': 'Appositions',
    'n_dependency_conj': 'Conjuncts',
    'n_dependency_amod': 'Adjectival modifiers',
    'n_dependency_advmod': 'Adverbial modifiers',
    'tree_depth': 'Dependency-tree depth',
    'tree_width': 'Dependency-tree width',
    'tree_branching': 'Dependency branching',
    'pos_variability': 'POS variability',
    'lexical_density': 'Lexical density',
    'ttr': 'Type–token ratio',
    'rttr': 'Root type–token ratio',
    'cttr': 'Corrected type–token ratio',
    'herdan_c': 'Herdan’s C',
    'giroud_index': 'Guiraud index',
    'mtld': 'MTLD',
    'mattr': 'MATTR',
    'msttr': 'MSTTR',
    'hdd': 'HD-D',
    'yule_k': 'Yule’s K',
    'simpsons_d': 'Simpson’s D',
    'sichel_s': 'Sichel’s S',
    'global_sichel_s': 'Global Sichel’s S',
    'n_hapax_legomena': 'Hapax legomena',
    'n_hapax_dislegomena': 'Hapax dislegomena',
    'n_global_token_hapax_legomena': 'Global token hapax',
    'n_global_lemma_hapax_legomena': 'Global lemma hapax',
    'n_global_token_hapax_dislegomena': 'Global token dislegomena',
    'n_global_lemma_hapax_dislegomena': 'Global lemma dislegomena',
    'n_hedges': 'Hedge words',
    'avg_n_synsets': 'Average synset count',
    'avg_n_synsets_noun': 'Average noun synsets',
    'avg_n_synsets_verb': 'Average verb synsets',
    'avg_n_synsets_adj': 'Average adjective synsets',
    'avg_n_synsets_adv': 'Average adverb synsets',
    'n_low_synsets': 'Low-synset words',
    'n_high_synsets': 'High-synset words',
    'n_low_synsets_noun': 'Low-synset nouns',
    'n_high_synsets_noun': 'High-synset nouns',
    'n_low_synsets_verb': 'Low-synset verbs',
    'n_high_synsets_verb': 'High-synset verbs',
    'n_positive_sentiment': 'Positive-sentiment words',
    'n_negative_sentiment': 'Negative-sentiment words',
    'sentiment_score': 'Sentiment score',
    'avg_valence': 'Average valence',
    'avg_arousal': 'Average arousal',
    'avg_dominance': 'Average lexical dominance',
    'n_low_valence': 'Low-valence words',
    'n_high_valence': 'High-valence words',
    'n_low_arousal': 'Low-arousal words',
    'n_high_arousal': 'High-arousal words',
    'n_low_dominance': 'Low-dominance words',
    'n_high_dominance': 'High-dominance words',
    'avg_intensity_anger': 'Average anger intensity',
    'n_low_intensity_anger': 'Low anger-intensity words',
    'n_high_intensity_anger': 'High anger-intensity words',
    'avg_concreteness': 'Average concreteness',
    'avg_sd_concreteness': 'Concreteness-rating variability',
    'n_low_concreteness': 'Low-concreteness words',
    'n_high_concreteness': 'High-concreteness words',
    'avg_socialness': 'Average socialness',
    'avg_sd_socialness': 'Socialness-rating variability',
    'n_low_socialness': 'Low-socialness words',
    'n_high_socialness': 'High-socialness words',
    'avg_Auditory_sensorimotor': 'Auditory associations',
    'avg_Interoceptive_sensorimotor': 'Interoceptive associations',
    'avg_Hand_arm_sensorimotor': 'Hand/arm associations',
    'avg_Mouth_sensorimotor': 'Mouth associations',
    'avg_Torso_sensorimotor': 'Torso associations',
    'avg_sd_Auditory_sensorimotor': 'Auditory-rating variability',
    'avg_sd_Interoceptive_sensorimotor': 'Interoceptive-rating variability',
    'avg_sd_Hand_arm_sensorimotor': 'Hand/arm-rating variability',
    'avg_sd_Mouth_sensorimotor': 'Mouth-rating variability',
    'avg_sd_Torso_sensorimotor': 'Torso-rating variability',
    'avg_iconicity': 'Average iconicity',
    'avg_sd_iconicity': 'Iconicity-rating variability',
    'flesch_reading_ease': 'Flesch Reading Ease',
    'flesch_kincaid_grade': 'Flesch–Kincaid grade',
    'gunning_fog': 'Gunning Fog index',
    'ari': 'Automated Readability Index',
    'lix': 'LIX readability',
    'compressibility': 'Compressibility',
    'entropy': 'Entropy',
}

# Fixed selection: independent of observed differences, shared across transcripts.
# Each tuple gives the original ELFEN column, display name, and plotted unit.
FIGURE1_FEATURES = [
    ('n_tokens', 'Turn length', 'Tokens per turn'),
    ('n_sentences', 'Sentence count', 'Sentences per turn'),
    ('tokens_per_sentence', 'Sentence length', 'Tokens per sentence'),
    ('n_PRON_Person_1', 'First-person pronouns', 'Per 100 tokens'),
    ('n_PRON_Person_2', 'Second-person pronouns', 'Per 100 tokens'),
    ('n_PRON_Person_3', 'Third-person pronouns', 'Per 100 tokens'),
    ('n_hedges', 'Hedge words', 'Per 100 tokens'),
    ('n_dependency_advcl', 'Adverbial clauses', 'Per 100 tokens'),
    ('n_dependency_ccomp', 'Clausal complements', 'Per 100 tokens'),
    ('n_adj', 'Adjectives', 'Per 100 tokens'),
    ('lexical_density', 'Lexical density', 'ELFEN density score'),
    ('n_positive_sentiment', 'Positive-sentiment words', 'Per 100 tokens'),
    ('n_negative_sentiment', 'Negative-sentiment words', 'Per 100 tokens'),
    ('avg_valence', 'Average lexical valence', 'Lexicon rating'),
    ('avg_arousal', 'Average lexical arousal', 'Lexicon rating'),
    ('avg_dominance', 'Average lexical dominance', 'Lexicon rating'),
    ('avg_intensity_anger', 'Average lexical anger intensity', 'Lexicon rating'),
    ('avg_concreteness', 'Average concreteness', 'Lexicon rating'),
    ('avg_socialness', 'Average socialness', 'Lexicon rating'),
    ('gunning_fog', 'Gunning Fog readability', 'Index: higher = harder'),
]
TOP_N_FEATURES = len(FIGURE1_FEATURES)  # Fixed 20, not a ranking by shift.
FIGURE1_METADATA = {}
figure1_additions = {}
figure1_token_count = pd.to_numeric(df_results['n_tokens'], errors='coerce')
figure1_denominator = figure1_token_count.where(figure1_token_count.gt(0))
for feature, label, unit in FIGURE1_FEATURES:
    column = feature + '_per_100_tokens' if unit == 'Per 100 tokens' else feature
    if column not in df_results.columns:
        if feature not in df_results.columns:
            raise ValueError(f'Missing selected ELFEN feature: {feature}')
        # New normalized columns only: retain all existing columns and values.
        figure1_additions[column] = (
            100 * pd.to_numeric(df_results[feature], errors='coerce') / figure1_denominator
        )
    FIGURE1_METADATA[column] = {'label': label, 'unit': unit}
if figure1_additions:
    df_results = pd.concat(
        [df_results, pd.DataFrame(figure1_additions, index=df_results.index)], axis=1
    ).copy()
FEATURE_DISPLAY_NAMES.update({f: label for f, label, unit in FIGURE1_FEATURES})


def readable_feature_name(feature):
    suffix = '_per_100_tokens'
    base_feature = feature[:-len(suffix)] if feature.endswith(suffix) else feature
    return FEATURE_DISPLAY_NAMES.get(base_feature, base_feature)


def calculate_phase_shifts(patient_data, feature_cols):
    groups = {
        phase: patient_data.loc[patient_data['phase'].eq(phase), feature_cols]
        .apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
        for phase in ['Baseline', 'Rupture', 'Repair']
    }
    counts = {phase: len(group) for phase, group in groups.items()}
    if counts['Baseline'] < 5:
        print('Small baseline group: standardized shifts are descriptive and may be unstable.')
    rows = []
    for feature in feature_cols:
        baseline = groups['Baseline'][feature].dropna()
        sd = baseline.std(ddof=1)
        estimable = len(baseline) >= 2 and np.isfinite(sd) and sd > 0
        # Keep constant features and small groups: raw means still carry information.
        row = {'Feature': feature, 'Baseline_Mean': baseline.mean(),
               'Baseline_SD': sd, 'N_Baseline': len(baseline)}
        for phase in ['Rupture', 'Repair']:
            values = groups[phase][feature].dropna()
            row[f'N_{phase}'] = len(values)
            row[f'{phase}_Mean'] = values.mean()
            row[f'Z_Shift_{phase}'] = (values.mean()-baseline.mean())/sd if estimable else np.nan
        rows.append(row)
    shifts = pd.DataFrame(rows)
    shifts['Max_Abs_Shift'] = shifts[['Z_Shift_Rupture', 'Z_Shift_Repair']].abs().max(axis=1)
    return shifts, counts


def make_phase_plot_data(shifts, top_n=20):
    # Retain function name/argument, but select in a fixed conceptual order.
    features = list(FIGURE1_METADATA)[:top_n]
    lookup = shifts.set_index('Feature')
    rows = []
    for feature in features:
        for phase in ['Baseline', 'Rupture', 'Repair']:
            row = lookup.loc[feature]
            rows.append({'Feature': feature,
                         'Feature_Display': FIGURE1_METADATA[feature]['label'],
                         'Unit': FIGURE1_METADATA[feature]['unit'],
                         'Phase': phase, 'Mean': row[f'{phase}_Mean'],
                         'N': int(row[f'N_{phase}']),
                         'Comparison': f'{phase} vs Baseline',
                         'Z_Shift': row.get(f'Z_Shift_{phase}', np.nan)})
    return pd.DataFrame(rows)


def export_phase_analysis(patient_data, feature_cols, destination, transcript_name, top_n=20):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    feature_cols = list(dict.fromkeys(list(feature_cols) + list(FIGURE1_METADATA)))
    shifts, counts = calculate_phase_shifts(patient_data, feature_cols)
    plot_data = make_phase_plot_data(shifts, top_n)
    # Standardized shifts remain available as supplementary data, not plotted bars.
    shifts.to_csv(destination / 'linguistic_shifts_summary.csv', index=False)
    plot_data.to_csv(destination / 'figure1_plot_data.csv', index=False)
    features = plot_data['Feature'].drop_duplicates().tolist()
    comparisons = ['Rupture'] + (['Repair'] if counts['Repair'] else [])
    colors = {'Rupture': '#C77732', 'Repair': '#8573A8'}
    lookup = shifts.set_index('Feature')
    fig, ax = plt.subplots(figsize=(13, 10))
    fig.subplots_adjust(left=.30, right=.70, top=.87, bottom=.17)
    y = np.arange(len(features))
    plotted_values = []
    for j, phase in enumerate(comparisons):
        values = lookup.loc[features, f'Z_Shift_{phase}'].to_numpy(dtype=float)
        offsets = np.zeros(len(features)) if len(comparisons)==1 else np.full(len(features), (j-.5)*.24)
        valid = np.isfinite(values)
        plotted_values.extend(values[valid])
        ax.hlines(y[valid]+offsets[valid], 0, values[valid], color=colors[phase], alpha=.4, linewidth=1.2)
        ax.scatter(values[valid], y[valid]+offsets[valid], color=colors[phase], s=42,
                   marker='o' if phase=='Rupture' else 'D', label=f'{phase} vs baseline', zorder=3)
        for i in np.flatnonzero(~valid):
            ax.text(.02, y[i]+offsets[i], 'Not estimable' if len(comparisons)==1 else f'{phase}: not estimable',
                    transform=ax.get_yaxis_transform(), va='center', fontsize=7, color='#666666')
    extent = max([abs(v) for v in plotted_values], default=1)
    extent = max(extent*1.12, .5)
    ax.set_xlim(-extent, extent)
    ax.axvline(0, color='#555555', linewidth=1, linestyle='--')
    display_labels = [FIGURE1_METADATA[f]['label'] +
                      (' *' if FIGURE1_METADATA[f]['unit']=='Per 100 tokens' else '') for f in features]
    ax.set_yticks(y, display_labels, fontsize=9)
    ax.set_ylim(len(features)-.4, -.6)
    ax.set_xlabel('Difference from baseline (baseline SD units)', fontsize=10, fontweight='bold', labelpad=12)
    ax.grid(axis='y', visible=False)
    ax.grid(axis='x', alpha=.2)
    # Subtle separators preserve the fixed conceptual feature ordering.
    for boundary in [2.5, 6.5, 10.5, 16.5, 18.5]:
        if boundary < len(features)-1:
            ax.axhline(boundary, color='#dddddd', linewidth=.7)
    phases = ['Baseline'] + comparisons
    ax.text(1.04, 1.025, 'Mean (valid n)', transform=ax.transAxes,
            fontsize=9, fontweight='bold', ha='left')
    ax.text(1.04, 1.005, ' / '.join(phases), transform=ax.transAxes,
            fontsize=8, ha='left')
    def format_mean(value, count):
        return f'{value:.2f} ({int(count)})' if np.isfinite(value) else 'NA (0)'
    for i, feature in enumerate(features):
        row = lookup.loc[feature]
        text = ' / '.join(format_mean(row[f'{phase}_Mean'], row[f'N_{phase}']) for phase in phases)
        ax.text(1.04, i, text, transform=ax.get_yaxis_transform(),
                ha='left', va='center', fontsize=8, clip_on=False)
    ax.text(0, -.095, 'Lower than baseline', transform=ax.transAxes, fontsize=9, ha='left')
    ax.text(1, -.095, 'Higher than baseline', transform=ax.transAxes, fontsize=9, ha='right')
    if len(comparisons)>1:
        ax.legend(loc='lower left', bbox_to_anchor=(0,1.055), frameon=False, fontsize=9)
    fig.suptitle('Figure 1: Patient Linguistic Differences Across Interaction Phases',
                 fontsize=14, fontweight='bold', y=.97)
    fig.text(.30, .92,
             f"Patient turns: baseline={counts['Baseline']}, rupture={counts['Rupture']}, repair={counts['Repair']}.",
             fontsize=9)
    fig.text(.02, .045,
             'Fixed feature selection; points show descriptive standardized differences, not significance tests. Small baseline SDs can inflate differences.\n'
             'Right column: phase means in original units, with valid turn counts. * Frequencies per 100 tokens.', fontsize=8)
    sns.despine(ax=ax)
    fig.savefig(destination/'figure1_linguistic_phase_shifts.png',dpi=300,bbox_inches='tight')
    plt.close(fig)
    # Individual values remain available for checking the summary, but are not plotted.
    individual_data = patient_data[['turn_id','phase'] + features].melt(
        id_vars=['turn_id','phase'], var_name='Feature', value_name='Value'
    ).rename(columns={'phase':'Phase'})
    individual_data.to_csv(destination/'figure1_individual_turn_data.csv', index=False)
    return shifts, plot_data


patient_df = df_results.loc[df_results['speaker'].eq('Patient')].copy()
shifts_df, plot_df = export_phase_analysis(
    patient_df, elfen_feature_cols, output_dir, excel_path.stem, TOP_N_FEATURES
)
# Preserve the feature set used by your subsequent correlation analysis.
valid_features = [feature for feature in elfen_feature_cols if feature in shifts_df['Feature'].tolist()]


In [18]:
# 5. FIGURE 2: Interpersonal positions and turn sequence
# ---------------------------------------------------------
required = ['turn_id', 'speaker', 'affiliation', 'dominance']
missing = [col for col in required if col not in df_results.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Rebuild plotting data from the current results on every execution.
coords = (
    df_results[required]
    .copy()
    .sort_values('turn_id')
    .reset_index(drop=True)
)

for col in ['affiliation', 'dominance']:
    coords[col] = pd.to_numeric(coords[col], errors='coerce')

coords[['affiliation', 'dominance']] = (
    coords[['affiliation', 'dominance']]
    .replace([np.inf, -np.inf], np.nan)
)

# Validate the assumed annotation scale.
for col in ['affiliation', 'dominance']:
    invalid = coords[col].notna() & ~coords[col].between(-1, 1)
    if invalid.any():
        raise ValueError(f"{col} contains ratings outside [-1, 1].")

styles = {
    'Patient': ('#d62728', 's'),
    'Therapist': ('#1f77b4', 'o'),
}
unknown = set(coords['speaker'].dropna()) - set(styles)
if unknown or coords['speaker'].isna().any():
    raise ValueError(f"Missing or unexpected speaker labels: {unknown}")

coords['plotted'] = (
    coords[['affiliation', 'dominance']].notna().all(axis=1)
)
coords['radius'] = np.hypot(
    coords['affiliation'], coords['dominance']
)
coords['angle_radians'] = np.arctan2(
    coords['dominance'], coords['affiliation']
)

# Include all turns in the export, including omitted/unrated turns.
coords['transcript'] = excel_path.stem
output_dir.mkdir(parents=True, exist_ok=True)
coords.to_csv(output_dir / 'figure2_plot_data.csv', index=False)

fig, ax = plt.subplots(
    figsize=(8, 8),
    subplot_kw={'projection': 'polar'},
    dpi=300,
)

ax.set_theta_zero_location('E')
ax.set_theta_direction(1)
ax.set_ylim(0, 1.55)  # Same scale across transcripts.

# Connect only adjacent turns with complete ratings.
# Do not bridge gaps created by missing ratings.
for i in range(1, len(coords)):
    previous = coords.iloc[i - 1]
    current = coords.iloc[i]

    if not (previous['plotted'] and current['plotted']):
        continue

    # Identical positions need no arrow.
    if (
        previous['affiliation'] == current['affiliation']
        and previous['dominance'] == current['dominance']
    ):
        continue

    ax.annotate(
        '',
        xy=(current['angle_radians'], current['radius']),
        xytext=(previous['angle_radians'], previous['radius']),
        arrowprops=dict(
            arrowstyle='->', color='gray',
            lw=0.8, ls='dotted', alpha=0.5,
        ),
        zorder=1,
    )

for speaker, (color, marker) in styles.items():
    points = coords[
        coords['plotted'] & coords['speaker'].eq(speaker)
    ]
    if not points.empty:
        ax.scatter(
            points['angle_radians'],
            points['radius'],
            color=color,
            marker=marker,
            s=85,
            label=speaker,
            zorder=3,
        )

# Group repeated positions to avoid stacking individual turn labels.
rated = coords[coords['plotted']]
for (speaker, affiliation, dominance), group in rated.groupby(
    ['speaker', 'affiliation', 'dominance'], sort=False
):
    prefix = 'P' if speaker == 'Patient' else 'T'
    turn_labels = [str(int(t)) for t in group['turn_id']]
    lines = [
        ', '.join(turn_labels[i:i + 5])
        for i in range(0, len(turn_labels), 5)
    ]
    label = f"{prefix}: " + '\n'.join(lines)
    offset = (7, 10) if speaker == 'Patient' else (7, -20)

    ax.annotate(
        label,
        (group['angle_radians'].iloc[0], group['radius'].iloc[0]),
        textcoords='offset points',
        xytext=offset,
        fontsize=7,
        color=styles[speaker][0],
    )

ax.set_xticks(np.radians([0, 90, 180, 270]))
ax.set_xticklabels([
    'Friendly\n(Affiliation +)',
    'Dominant\n(Control +)',
    'Hostile\n(Affiliation −)',
    'Submissive\n(Control −)',
])

if not rated.empty:
    ax.legend(bbox_to_anchor=(1.15, 1.1), loc='upper left')
else:
    ax.text(
        0.5, 0.5, 'No complete interpersonal ratings',
        transform=ax.transAxes, ha='center',
    )

omitted = int((~coords['plotted']).sum())
ax.set_title(
    f'Figure 2: Interpersonal Circumplex Trajectory Across Session Turns',
    fontweight='bold'
)
#fig.text(
#    0.02, 0.02,
#    f'{omitted} turns omitted because ratings are incomplete.\n'
#    'Arrows connect consecutive rated turns across speakers.',
#    fontsize=8,
#)

fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(
    output_dir / 'figure2_interpersonal_circumplex.png',
    bbox_inches='tight',
    dpi=300,
)
plt.close(fig)

print(f"Figure 2 and coordinate CSV saved to: {output_dir}")

Figure 2 and coordinate CSV saved to: /mount/arbeitsdaten/e-delib/vecchiea/data/answering_machine/transcript_alcohol_mistral_humanrating_discourse


In [19]:
# 6. FIGURE 3: Sequential Rupture-Repair Timeline
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 3.5), dpi=300)

phase_map = {'Baseline': 0, 'Withdrawal': -1, 'Confrontation': -2, 'Repair': 1}
y_states = df_results['interaction_state'].map(phase_map).to_numpy()

ax.plot(df_results['turn_id'], y_states, color='#2b5c8f', marker='o', linewidth=2, markersize=7)
ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)

ax.set_yticks([-2, -1, 0, 1])
ax.set_yticklabels([
    'Confrontation Rupture', 
    'Withdrawal Rupture', 
    'Collaborative Baseline', 
    'Repair / Metacommunication'
])

ax.set_xlabel("Turn Sequence ID", fontweight='bold')
ax.set_ylabel("Interactional State", fontweight='bold')
ax.set_title("Figure 3: Sequential Dynamics of Ruptures and Metacommunication Repairs", fontweight='bold')
ax.grid(True, linestyle=":", alpha=0.6)

for i, row in df_results.iterrows():
    ax.annotate(
        row['speaker'][0], # 'T' or 'P'
        (row['turn_id'], y_states[i]),
        textcoords="offset points",
        xytext=(0, 8),
        ha='center',
        fontsize=8,
        style='italic'
    )

ax.set_ylim(-2.5, 1.5)
sns.despine(top=True, right=True)
plt.tight_layout()
plt.savefig(output_dir / "figure3_rupture_repair_timeline.png", bbox_inches='tight', dpi=300)
plt.close()

In [20]:
# 7. Correlations & CSV Exports
# ---------------------------------------------------------
corr_results = []
for feat in valid_features:
    aff_corr = patient_df[feat].corr(patient_df['affiliation'])
    dom_corr = patient_df[feat].corr(patient_df['dominance'])
    corr_results.append({
        'Feature': feat,
        'Corr_Affiliation': aff_corr,
        'Corr_Dominance': dom_corr
    })

corr_df = pd.DataFrame(corr_results)

# The phase summary is exported together with Figure 1 above.
corr_df.to_csv(output_dir / "linguistic_annotation_correlations.csv", index=False)
df_results.to_csv(output_dir / "full_discourse_analysis_output.csv", index=False)

print(f"\nAnalysis complete! Figures and tables saved to:\n{output_dir}")


Analysis complete! Figures and tables saved to:
/mount/arbeitsdaten/e-delib/vecchiea/data/answering_machine/transcript_alcohol_mistral_humanrating_discourse
